# 07 — End-to-End Threat Report Analyzer

**Inputs:** trained components from Notebooks 3–5  
**Outputs:** `artifacts/models/*.pkl`, `artifacts/models/*.json`, per-report JSON  
**Feeds thesis:** Chapter 4.4 — End-to-End Threat Report Analysis

This notebook turns the pipeline into a working analyst tool:

1. Loads cached models (hybrid classifier + BERTopic).
2. Extracts text from a PDF or TXT report.
3. Classifies every sentence as relevant / irrelevant.
4. Assigns relevant sentences to topics.
5. Applies three emerging-threat rules.
6. Renders a **CSS-styled verdict card** directly in the notebook output.


## 1. Setup

In [46]:
import pandas as pd, numpy as np, json, os, re, joblib
from pathlib import Path
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import hstack
from IPython.display import HTML, display

os.makedirs('../artifacts/models', exist_ok=True)
print('ready')

ready


## 2. Save trained models (one-time)

Retrains the hybrid classifier on the training set (<1 s) and saves all four
artifacts needed for inference.

In [47]:
train = pd.read_csv('../artifacts/train.csv')
X_train_emb = np.load('../artifacts/train_emb.npy')

vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_tfidf = vec.fit_transform(train['sentence_text'])
X_hybrid = hstack([X_tfidf, normalize(X_train_emb)]).tocsr()
clf = LogisticRegression(C=2.0, class_weight='balanced',
                         max_iter=2000, solver='liblinear', random_state=42)
clf.fit(X_hybrid, train['relevant'])

joblib.dump(vec, '../artifacts/models/tfidf_vectorizer.pkl')
joblib.dump(clf, '../artifacts/models/hybrid_classifier.pkl')
print('saved classifier + vectorizer')

rel = pd.read_csv('../artifacts/rel_with_topics.csv')
baseline = rel['topic'].value_counts(normalize=True).to_dict()
baseline = {int(k): float(v) for k, v in baseline.items()}
json.dump(baseline, open('../artifacts/models/topic_baseline.json','w'), indent=2)
print('saved topic baseline:', len(baseline), 'entries')

from collections import Counter
tokens = re.findall(r'\b[a-z]{3,}\b', ' '.join(train['sentence_text'].str.lower()))
vocab = set(w for w, _ in Counter(tokens).most_common(50000))
json.dump(sorted(vocab), open('../artifacts/models/vocab_reference.json','w'))
print('saved vocab reference:', len(vocab), 'tokens')

saved classifier + vectorizer
saved topic baseline: 25 entries
saved vocab reference: 11611 tokens


## 3. Load the BERTopic model

In [48]:
topic_model = BERTopic.load('../artifacts/bertopic_model')
print('loaded BERTopic model with', len(topic_model.get_topics())-1, 'topics')

2026-10-18 17:05:03,834 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


loaded BERTopic model with 24 topics


## 4. PDF / text extraction

Requires `pdfplumber` for PDF files. Plain text works with no extra dependency.

In [49]:
def extract_text(path):
    p = Path(path)
    if p.suffix.lower() == '.pdf':
        import pdfplumber
        with pdfplumber.open(p) as pdf:
            return '\n'.join(page.extract_text() or '' for page in pdf.pages)
    return p.read_text(encoding='utf-8', errors='ignore')

def split_sentences(text, min_words=5):
    text = re.sub(r'\s+', ' ', text)
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in parts if len(s.split()) >= min_words]

## 5. `predict_report(path)`

Full pipeline: extract → classify → assign topics → apply emerging-threat rules.

In [50]:
def predict_report(path):
    raw = extract_text(path)
    sentences = split_sentences(raw)
    if not sentences:
        return {'report': str(path), 'error': 'no sentences extracted'}

    # ── 1. Classification ────────────────────────────────
    emb = SBERT.encode(sentences, batch_size=64, convert_to_numpy=True)
    X = hstack([vec.transform(sentences), normalize(emb)]).tocsr()
    preds = clf.predict(X)

    relevant_sents = [s for s, p in zip(sentences, preds) if p == 1]
    irrelevant_sents = [s for s, p in zip(sentences, preds) if p == 0]
    relevant_emb = emb[preds == 1]

    classification = {
        'total_sentences': len(sentences),
        'relevant_sentences': len(relevant_sents),
        'irrelevant_sentences': len(irrelevant_sents),
        'relevant_fraction': round(len(relevant_sents) / len(sentences), 3),
        'top_relevant_examples': relevant_sents[:3],   # for inspection
    }

    # ── 2. Topic assignment on relevant sentences ─────────
    if len(relevant_sents) > 0:
        topics, _ = topic_model.transform(relevant_sents, relevant_emb)
    else:
        topics = []

    topic_counts = {}
    for t in topics:
        t = int(t)
        topic_counts[t] = topic_counts.get(t, 0) + 1

    # ── 3. Emerging-threat detection ─────────────────────
    reasons, emerging = [], False
    rule_hits = {'outlier': False, 'surge': False, 'novelty': False}

    if len(topics) > 0 and -1 in topic_counts:
        frac = topic_counts[-1] / len(topics)
        if frac > 0.30:
            emerging = True
            rule_hits['outlier'] = True
            reasons.append(f"{topic_counts[-1]} sentences ({frac*100:.0f}%) "
                           f"could not be assigned to any known topic")

    for tid, count in topic_counts.items():
        if tid == -1:
            continue
        base = TOPIC_BASE.get(tid, 1e-9)
        surge = (count / len(topics)) / max(base, 1e-9)
        if surge > 3.0 and count >= 3:
            emerging = True
            rule_hits['surge'] = True
            reasons.append(f"topic {tid} appears {count}x "
                           f"({surge:.1f}x historical baseline)")

    toks = [t for s in relevant_sents
            for t in re.findall(r'\b[a-z]{3,}\b', s.lower())]
    novelty_ratio = 0.0
    if toks:
        novel = [t for t in toks if t not in VOCAB]
        novelty_ratio = len(novel) / len(toks)
        if novelty_ratio > 0.40:
            emerging = True
            rule_hits['novelty'] = True
            reasons.append(f"{novelty_ratio*100:.0f}% of tokens are novel")

    # ── 4. Dominant topic ────────────────────────────────
    dominant = None
    if topic_counts:
        dom_tid = max(topic_counts, key=topic_counts.get)
        if dom_tid != -1:
            dominant = {
                'topic_id': dom_tid,
                'top_words': [w for w, _ in topic_model.get_topic(dom_tid)][:8],
                'count': topic_counts[dom_tid],
                'share': round(topic_counts[dom_tid] / len(topics), 3),
            }
            reasons.append(f"dominant topic {dom_tid}: "
                           + ', '.join(dominant['top_words']))

    # ── 5. Verdict ───────────────────────────────────────
    if not relevant_sents:
        verdict = 'not_relevant'
    elif emerging:
        verdict = 'emerging'
    else:
        verdict = 'known'

    # ── 6. Combined output ───────────────────────────────
    return {
        'report': str(path),

        # classification view
        'classification': classification,

        # emerging-detection view
        'emerging_detection': {
            'emerging_threat': emerging,
            'topic_counts': {str(k): v for k, v in topic_counts.items()},
            'dominant_topic': dominant,
            'novelty_ratio': round(novelty_ratio, 3),
            'rule_hits': rule_hits,
            'reasons': reasons,
        },

        # top-level verdict (for quick access)
        'verdict': verdict,
        # flattened for backward compatibility with existing cells
        'total_sentences': classification['total_sentences'],
        'relevant_sentences': classification['relevant_sentences'],
        'relevant_fraction': classification['relevant_fraction'],
        'topic_counts': {str(k): v for k, v in topic_counts.items()},
        'dominant_topic': dominant,
        'emerging_threat': emerging,
        'reasons': reasons,
    }

## 6. Run on an inline CTI sample

Quick sanity test using text with recognisable CTI terms.

In [51]:
sample_text = '''
The threat actor exploited CVE-2023-23397 to gain initial access via Outlook.
A malicious DLL named glib-2.0.dll was dropped in the %PUBLIC% folder.
The loader contacted a hardcoded C2 server at 185.220.101.15 over an SSL socket.
Credentials were harvested from compromised cloud accounts using a phishing kit.
Ransomware activity was observed against the subsidiary, with the CyberVolk family encrypting files.
North Korean actors were attributed to a separate campaign targeting financial institutions.
'''

test_path = '../artifacts/test_sample.txt'
open(test_path, 'w').write(sample_text)

result = predict_report(test_path)
print(json.dumps(result, indent=2))

2026-10-18 17:05:04,454 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


{
  "report": "../artifacts/test_sample.txt",
  "classification": {
    "total_sentences": 6,
    "relevant_sentences": 5,
    "irrelevant_sentences": 1,
    "relevant_fraction": 0.833,
    "top_relevant_examples": [
      "The threat actor exploited CVE-2023-23397 to gain initial access via Outlook.",
      "The loader contacted a hardcoded C2 server at 185.220.101.15 over an SSL socket.",
      "Credentials were harvested from compromised cloud accounts using a phishing kit."
    ]
  },
  "emerging_detection": {
    "emerging_threat": false,
    "topic_counts": {
      "3": 1,
      "2": 1,
      "4": 1,
      "6": 1,
      "5": 1
    },
    "dominant_topic": {
      "topic_id": 3,
      "top_words": [
        "zeroday",
        "exploitation",
        "in",
        "vulnerabilities",
        "vulnerability",
        "exploited",
        "exploit",
        "2024"
      ],
      "count": 1,
      "share": 0.2
    },
    "novelty_ratio": 0.02,
    "rule_hits": {
      "outlier": false,

## 7. Render the verdict as a CSS card

The output below is styled HTML rendered directly inside Jupyter — suitable for
screenshotting into the thesis or presenting live in the viva.

In [52]:
VERDICT_STYLES = {
    'emerging':     {'bg': '#fdecea', 'border': '#c0392b', 'label': 'EMERGING THREAT SIGNAL', 'icon': '⚠'},
    'known':        {'bg': '#eafaf1', 'border': '#27ae60', 'label': 'KNOWN THREAT PATTERN',    'icon': '✔'},
    'not_relevant': {'bg': '#f0f0f0', 'border': '#7f8c8d', 'label': 'NOT RELEVANT',           'icon': '—'},
}

def render_verdict(result):
    v = result.get('verdict', 'not_relevant')
    s = VERDICT_STYLES.get(v, VERDICT_STYLES['not_relevant'])
    report_name = Path(result['report']).name

    # -- summary rows --
    rows = [
        ('Total sentences',   result['total_sentences']),
        ('Relevant sentences', result['relevant_sentences']),
        ('Relevant fraction', f"{result['relevant_fraction']*100:.1f}%"),
        ('Emerging threat',   'Yes' if result['emerging_threat'] else 'No'),
    ]
    summary_html = ''.join(
        f'<tr><td style="padding:6px 12px;border-bottom:1px solid #eee;color:#555;">{k}</td>'
        f'<td style="padding:6px 12px;border-bottom:1px solid #eee;font-weight:600;">{v}</td></tr>'
        for k, v in rows
    )

    # -- dominant topic card --
    dom = result.get('dominant_topic')
    if dom:
        words_html = ' '.join(
            f'<span style="display:inline-block;background:#1f3a93;color:#fff;'
            f'padding:3px 8px;border-radius:12px;margin:2px;font-size:11px;">{w}</span>'
            for w in dom['top_words']
        )
        dom_html = f'''
        <div style="margin-top:16px;">
          <div style="font-size:13px;color:#555;margin-bottom:8px;">
            Dominant topic <b>#{dom['topic_id']}</b> — appears in <b>{dom['count']}</b> relevant sentences
          </div>
          <div>{words_html}</div>
        </div>
        '''
    else:
        dom_html = ''

    # -- topic distribution --
    tc = result.get('topic_counts', {})
    if tc:
        tc_items = sorted(tc.items(), key=lambda x: -int(x[1]))
        tc_html = ' '.join(
            f'<span style="display:inline-block;background:#ecf0f1;color:#2c3e50;'
            f'padding:3px 10px;border-radius:12px;margin:2px;font-size:11px;">'
            f'T{k} <b>{v}</b></span>'
            for k, v in tc_items
        )
        tc_section = f'''
        <div style="margin-top:16px;">
          <div style="font-size:13px;color:#555;margin-bottom:8px;">Topic distribution</div>
          <div>{tc_html}</div>
        </div>
        '''
    else:
        tc_section = ''

    # -- reasons --
    if result.get('reasons'):
        reasons_html = ''.join(
            f'<li style="margin:4px 0;color:#333;">{r}</li>'
            for r in result['reasons']
        )
        reasons_section = f'''
        <div style="margin-top:16px;">
          <div style="font-size:13px;color:#555;margin-bottom:6px;">Why this verdict</div>
          <ul style="margin:0;padding-left:20px;">{reasons_html}</ul>
        </div>
        '''
    else:
        reasons_section = ''

    html = f'''
    <div style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;max-width:720px;
                border:1px solid #ddd;border-radius:10px;padding:20px;
                box-shadow:0 2px 6px rgba(0,0,0,0.06);background:#fafbfc;">

      <div style="font-size:18px;font-weight:700;color:#1f3a93;margin-bottom:4px;">
        CTI Threat Report Analysis
      </div>
      <div style="font-size:12px;color:#888;margin-bottom:16px;">
        Report: <b>{report_name}</b>
      </div>

      <div style="background:{s['bg']};border-left:6px solid {s['border']};
                  padding:14px 18px;border-radius:6px;font-size:16px;
                  font-weight:700;color:{s['border']};margin-bottom:16px;">
        {s['icon']} &nbsp; {s['label']}
      </div>

      <div style="background:#fff;border-radius:6px;padding:8px 4px;">
        <table style="border-collapse:collapse;width:100%;font-size:13px;">
          {summary_html}
        </table>
      </div>

      {dom_html}
      {tc_section}
      {reasons_section}

      <div style="margin-top:24px;padding-top:12px;border-top:1px solid #eee;
                  font-size:10px;color:#aaa;text-align:center;">
        Generated by CTI Report Analyzer — Hybrid TF-IDF + SBERT classifier with BERTopic
      </div>
    </div>
    '''
    display(HTML(html))

render_verdict(result)

Total sentences,6
Relevant sentences,5
Relevant fraction,83.3%
Emerging threat,No


## 8. Analyse a real PDF report (optional)

Point `PDF_PATH` at any PDF from the SMAH-CTI source collection (APTNotes, CrowdStrike, Mandiant, MITRE, Microsoft). If the file is not present, this cell prints a skip message — everything else still works.

In [53]:
PDF_PATH = '../dataset/sample_report.pdf'
if os.path.exists(PDF_PATH):
    result_pdf = predict_report(PDF_PATH)
    print(json.dumps(result_pdf, indent=2))
    json.dump(result_pdf, open('../artifacts/report_analysis.json','w'), indent=2)
    print('saved -> artifacts/report_analysis.json')
    render_verdict(result_pdf)
else:
    print(f'No PDF at {PDF_PATH} — skipping. Put a CTI PDF there to test.')

No PDF at ../dataset/sample_report.pdf — skipping. Put a CTI PDF there to test.


## 9. How to analyse a new report

After this notebook runs once, the trained artifacts are saved. Analyse any
report from the command line:

```bash
python scripts/predict_report.py path/to/new_report.pdf
```

Or, inside Jupyter, use the `predict_report()` and `render_verdict()` functions
defined in Cells 5 and 7.